In [2]:
import pandas as pd

#
df_csv = pd.read_csv('/Users/priya/Documents/Projects/Accident/accident.csv',sep=";")
#print(df_csv.head())
print(df_csv.columns)


Index(['DATE', 'TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE', 'LONGITUDE',
       'ON STREET NAME', 'NUMBER OF PEDESTRIANS INJURED',
       'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED',
       'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED',
       'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1',
       'CONTRIBUTING FACTOR VEHICLE 2', 'CONTRIBUTING FACTOR VEHICLE 3',
       'CONTRIBUTING FACTOR VEHICLE 4', 'CONTRIBUTING FACTOR VEHICLE 5',
       'COLLISION_ID', 'VEHICLE TYPE CODE 1', 'VEHICLE TYPE CODE 2',
       'VEHICLE TYPE CODE 3', 'VEHICLE TYPE CODE 4', 'VEHICLE TYPE CODE 5'],
      dtype='object')


In [3]:
import json

json_file = "borough_data.json"
with open(json_file) as f:
    borough_data = json.load(f)

print(borough_data)


{'the bronx': {'name': 'the bronx', 'population': 1471160.0, 'area': 42.1}, 'brooklyn': {'name': 'brooklyn', 'population': 2648771.0, 'area': 70.82}, 'manhattan': {'name': 'manhattan', 'population': 1664727.0, 'area': 22.83}, 'queens': {'name': 'queens', 'population': 2358582.0, 'area': 108.53}, 'staten island': {'name': 'staten island', 'population': 479458.0, 'area': 58.37}}


In [4]:
from pandas import json_normalize

json_file = pd.json_normalize(borough_data)
print(json_file.head())


  the bronx.name  the bronx.population  the bronx.area brooklyn.name  \
0      the bronx             1471160.0            42.1      brooklyn   

   brooklyn.population  brooklyn.area manhattan.name  manhattan.population  \
0            2648771.0          70.82      manhattan             1664727.0   

   manhattan.area queens.name  queens.population  queens.area  \
0           22.83      queens          2358582.0       108.53   

  staten island.name  staten island.population  staten island.area  
0      staten island                  479458.0               58.37  


In [ ]:
boroughs = ['the bronx', 'brooklyn', 'manhattan', 'queens', 'staten island']

# Create a list to hold each borough’s info
data_list = []

for b in boroughs:
    data_list.append({
        'borough': b.title().replace('The ', ''),  # makes it "Bronx", "Brooklyn", etc.
        'population': json_file[f'{b}.population'][0],
        'area': json_file[f'{b}.area'][0]
    })

# Create new borough DataFrame
df_borough = pd.DataFrame(data_list)
print(df_borough)


         borough  population    area
0          Bronx   1471160.0   42.10
1       Brooklyn   2648771.0   70.82
2      Manhattan   1664727.0   22.83
3         Queens   2358582.0  108.53
4  Staten Island    479458.0   58.37


In [ ]:
df_csv.columns = df_csv.columns.str.strip().str.lower().str.replace(' ', '_')
print(df_csv.head())
#df_merged = pd.merge(df_csv, df_borough, how='left', on='borough_name')
df_merged = pd.merge(
    df_csv,
    df_borough,
    how='left',
    left_on='borough',
    right_on='borough'
)

df_csv.dtypes


         date   time   borough  zip_code   latitude  longitude  \
0  09/26/2018  12:12     BRONX   10454.0  40.808987 -73.911316   
1  09/25/2018  16:30  BROOKLYN   11236.0  40.636005 -73.912510   
2  08/22/2019  19:30    QUEENS   11101.0  40.755490 -73.939530   
3  09/23/2018  13:10    QUEENS   11367.0        NaN        NaN   
4  08/20/2019  22:40     BRONX   10468.0  40.868336 -73.901270   

                     on_street_name  number_of_pedestrians_injured  \
0                               NaN                              0   
1  FLATLANDS AVENUE                                              1   
2                               NaN                              0   
3  MAIN STREET                                                   0   
4                               NaN                              0   

   number_of_pedestrians_killed  number_of_cyclist_injured  ...  \
0                             0                          0  ...   
1                             0                 

NameError: name 'df' is not defined

In [31]:
print(df_merged[['borough', 'population', 'area']].head())


    borough  population  area
0     BRONX         NaN   NaN
1  BROOKLYN         NaN   NaN
2    QUEENS         NaN   NaN
3    QUEENS         NaN   NaN
4     BRONX         NaN   NaN


In [35]:
df_merged.head()
df_csv.shape, df_borough.shape, df_merged.shape
df_merged[df_merged['population'].isna()][['borough']].sum()



borough    BRONXBROOKLYNQUEENSQUEENSBRONXQUEENSQUEENSBRON...
dtype: object

In [39]:
df_merged['date'] = pd.to_datetime(df_merged['date'])

df_merged['month'] = df_merged['date'].dt.month_name()  # e.g., 'January'
df_merged['year'] = df_merged['date'].dt.year            # e.g., 2024
